# Multiple Linear Regression Analysis (cleaned)

This notebook performs data loading, feature encoding, VIF (multicollinearity) checks, OLS modeling using statsmodels, assumption diagnostics (Residuals vs Fitted, Q-Q), and provides a final linear equation and coefficient interpretation.

Notes:
- I removed corrupted/large output metadata that prevented the notebook JSON from parsing on GitHub.
- The analysis cells are intact and runnable; run the notebook locally or in Colab to generate figures and full outputs.

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# Load data (ensure 'marketing_sales_data.csv' is present in the repository)
df = pd.read_csv('marketing_sales_data.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# Basic checks
print('Missing values per column:
', df.isnull().sum())
print('
Data types:
', df.dtypes)
df.describe()

In [ ]:
# One-hot encode categorical variables (drop_first=True to avoid dummy trap)
df_enc = pd.get_dummies(df, columns=['TV','Influencer'], drop_first=True)
df_enc.head()

In [ ]:
# Prepare features and target
y = df_enc['Sales']
X = df_enc.drop(columns=['Sales'])
# Keep column order for readability
X = X.reindex(sorted(X.columns), axis=1)
X.head()

In [ ]:
# VIF calculation to check multicollinearity
X_const = sm.add_constant(X)
vif = pd.DataFrame()
vif['feature'] = X_const.columns
vif['VIF'] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
vif

In [ ]:
# Split into train/test for evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

In [ ]:
# Fit OLS model on training data
model = sm.OLS(y_train, X_train_const).fit()
print(model.summary())

In [ ]:
# Predictions and basic metrics on test
y_pred = model.predict(X_test_const)
print('Test R^2:', r2_score(y_test, y_pred))
print('Test RMSE:', np.sqrt(mean_squared_error(y_test, y_pred)))

In [ ]:
# Residuals vs Fitted plot to assess non-linearity and heteroscedasticity
fitted = model.fittedvalues
residuals = model.resid
plt.figure(figsize=(8,5))
plt.scatter(fitted, residuals, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted')
plt.show()

In [ ]:
# Q-Q plot for residual normality
sm.qqplot(residuals, line='45', fit=True)
plt.title('Normal Q-Q')
plt.show()

In [ ]:
# Breusch-Pagan test for heteroscedasticity
bp_test = het_breuschpagan(residuals, X_train_const)
labels = ['LM stat', 'LM p-value', 'F stat', 'F p-value']
print(dict(zip(labels, bp_test)))

In [ ]:
# Coefficients and final linear equation (using model params)
params = model.params
print('Model coefficients:
', params)
# Construct human-readable equation
intercept = params.get('const', 0.0)
terms = []
for name, coeff in params.items():
    if name == 'const':
        continue
    terms.append(f'({coeff:.4f})*{name}')
equation = 'Sales = ' + f'{intercept:.4f}' + ' + ' + ' + '.join(terms)
print('
Estimated linear equation:
', equation)

## Interpretation guidance
- Each coefficient shows the expected change in Sales when that feature increases by one unit, holding other features constant.
- For one-hot encoded categorical variables (e.g., TV_Medium), the coefficient represents the difference relative to the dropped baseline category (e.g., TV_Low if drop_first=True).
- Use the p-values from model.summary() to decide which coefficients are statistically significant (commonly p < 0.05).

Next steps after verifying the notebook: run it to produce the Residuals vs Fitted and Q-Q plots and confirm the final linear equation and coefficient interpretations.